### Test volume access.

In [0]:
### All needed imports
import requests
import json
from pyspark.sql import Row

from pyspark.sql.functions import col, avg, round, count



In [0]:
csv_path = "/Volumes/dbr_dev/valerii-matviiv/top_universities/QS World University Rankings 2025 (Top global universities).csv"

df_rankings = spark.read.csv(csv_path, header=True, inferSchema=True)

df_rankings.write.format("delta").mode("overwrite").saveAsTable("`dbr_dev`.`valerii-matviiv`.qs_world_rankings")

display(spark.table("`dbr_dev`.`valerii-matviiv`.qs_world_rankings"))

### Test API Connection.

In [0]:
api_url = "http://universities.hipolabs.com/search?country=Poland"
response = requests.get(api_url)
data_list = json.loads(response.text)

spark_rows = []
for uni in data_list:
    web_page = uni.get("web_pages")[0] if uni.get("web_pages") else None
    domain = uni.get("domains")[0] if uni.get("domains") else None
    
    spark_rows.append(
        Row(
            Institution_Name_API=uni.get("name"),
            domain=domain,
            web_page=web_page
        )
    )

df_api = spark.createDataFrame(spark_rows)

df_api.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("`dbr_dev`.`valerii-matviiv`.api_universities")

display(spark.table("`dbr_dev`.`valerii-matviiv`.api_universities"))

### Basic spark methods.

In [0]:
df_qs = spark.table("`dbr_dev`.`valerii-matviiv`.qs_world_rankings")
df_api = spark.table("`dbr_dev`.`valerii-matviiv`.api_universities")

df_qs_poland = df_qs.select("Institution_Name", "RANK_2025", "RANK_2024", "SIZE", "Academic_Reputation_Score") \
                    .filter(col("Location") == "Poland")

print("1. Filtered QS Poland Universities:")
display(df_qs_poland)

df_joined = df_qs_poland.join(df_api, df_qs_poland["Institution_Name"] == df_api["Institution_Name_API"], "inner")

print("2. Joined Data (QS + API):")
display(df_joined)

df_grouped = df_joined.groupBy("SIZE") \
                      .agg(
                          round(avg("Academic_Reputation_Score"), 2).alias("avg_academic_score"),
                          count("Institution_Name").alias("university_count")
                      )

print("3. Aggregated Summary (GroupBy):")
display(df_grouped)